<a href="https://colab.research.google.com/github/sadineniManushree/flyrank--internship__ml/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sadineniManushree/flyrank--internship__ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


1. Two Paper Findings + My Methodology Questions

**Finding 1: "random_forest is the best model, selected by precision_at_50
(0.740), beating baseline_rules (0.240) and logistic_regression (0.400)."**

- Where does the label come from? The target is `is_declining_label`, which
  appears to be a real observed outcome (a traffic/engagement trend), not a
  product-decision column — the right kind of label to use. I'd ask the
  paper to state explicitly which raw columns the label was derived from, so
  a reader can independently verify it isn't circular.
- Does the validation design carry the claim? The paper states a
  `client_holdout` split was used, which is the honest choice for this data.
  My constructive question: were the reported metrics computed only on
  held-out test clients, or could training-client rows have leaked into the
  "Top 10 Queue Preview" shown later? A one-line confirmation would make the
  claim fully verifiable.
  **Finding 2: "Top features are days_with_impressions (0.158),
log_impressions_90d (0.128), and avg_position (0.109)."**

- Where does the label come from? Presumably the same trend-based label as
  above. Worth stating explicitly next to the feature importance table.
- Does the validation design carry the claim? days_with_impressions leads
  the list, but only modestly (about 1.23x the second-ranked feature) — not
  dramatically dominant, so this doesn't show the classic "one feature
  towers over everything" leakage red flag. My constructive question is
  smaller here: are days_with_impressions and log_impressions_90d
  measuring genuinely different information, or largely the same
  underlying signal counted two ways? If highly correlated, reporting them
  as two separate top features may overstate how many independent signals
  are actually driving the model.

In [61]:
# No computation needed for this section — findings are quoted directly from
# the FlyRank Refresh Opportunity Model Report's Model Comparison and Top
# Features tables. This cell just confirms the two numbers referenced above.
paper_best_precision_at_50 = 0.740
paper_baseline_precision_at_50 = 0.240
paper_top_feature_importance = 0.1578  # days_with_impressions
paper_second_feature_importance = 0.1282  # log_impressions_90d
print("Finding 1 gap:", paper_best_precision_at_50 - paper_baseline_precision_at_50)
print("Finding 2 dominance ratio:", round(paper_top_feature_importance / paper_second_feature_importance, 2))

Finding 1 gap: 0.5
Finding 2 dominance ratio: 1.23


In [62]:
from sklearn.metrics import roc_auc_score
print("Base rate (majority class):", round(max(y_test.mean(), 1 - y_test.mean()), 3))
print("Random Forest precision_at_50:", round(precision_at_k(y_test, proba_honest, 50), 3))
print("Baseline precision_at_50:", round(precision_at_k(y_test, base_scores, 50), 3))

Base rate (majority class): 0.609
Random Forest precision_at_50: 0.82
Baseline precision_at_50: 0.28


In [63]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(scores)[::-1][:k]
    return y_true.values[order].mean() if len(y_true) >= k else np.nan

In [49]:
from sklearn.ensemble import RandomForestClassifier

feature_cols = ['ctr', 'avg_position', 'impressions_90d', 'clicks_90d',
                 'engagement_rate', 'scroll_rate', 'search_volume', 'content_age_days']

X_train = df.loc[train_idx, feature_cols].fillna(0)
y_train = df.loc[train_idx, 'target']
X_test = df.loc[test_idx, feature_cols].fillna(0)
y_test = df.loc[test_idx, 'target']

rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25, class_weight='balanced_subsample', random_state=42)
rf_model.fit(X_train, y_train)
proba_honest = rf_model.predict_proba(X_test)[:, 1]
base_scores = df.loc[test_idx, 'baseline_score'].values

print("Random Forest and baseline scores ready.")

Random Forest and baseline scores ready.


In [50]:
from sklearn.model_selection import train_test_split

def client_aware_split(frame, target_col='target', client_col='client_id', random_state=42):
    clients = frame[client_col].fillna('unknown').astype(str)
    unique_clients = clients.drop_duplicates().to_numpy()
    if len(unique_clients) >= 5:
        rng = np.random.default_rng(random_state)
        shuffled = rng.permutation(unique_clients)
        n_test = max(1, int(round(len(shuffled) * 0.2)))
        test_clients = set(shuffled[:n_test])
        test_mask = clients.isin(test_clients).to_numpy()
        train_idx = frame.index[~test_mask]
        test_idx = frame.index[test_mask]
        if frame.loc[train_idx, target_col].nunique() == 2 and frame.loc[test_idx, target_col].nunique() == 2:
            return train_idx, test_idx, "client_holdout"
    train_idx, test_idx = train_test_split(frame.index, test_size=0.2, random_state=random_state, stratify=frame[target_col])
    return train_idx, test_idx, "stratified_row_holdout"

train_idx, test_idx, split_strategy = client_aware_split(df)
print("Split:", split_strategy, "| train:", len(train_idx), "| test:", len(test_idx))

Split: client_holdout | train: 27675 | test: 2325


In [51]:
df['target'] = (df['trend_direction'].astype(str).str.lower() == 'down').astype(int)
df['baseline_score'] = (df.groupby('position_tier')['ctr'].transform('mean') - df['ctr']) * df['impressions_90d']

In [52]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(len(df), "rows")

30000 rows


In [53]:
!git clone https://github.com/sadineniManushree/flyrank--internship__ml.git
%cd flyrank--internship__ml

Cloning into 'flyrank--internship__ml'...
remote: Enumerating objects: 165, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 165 (delta 70), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (165/165), 1.87 MiB | 13.04 MiB/s, done.
Resolving deltas: 100% (70/70), done.
/content/flyrank--internship__ml/flyrank--internship__ml/flyrank--internship__ml/flyrank--internship__ml


In [54]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My Model Under an Honest Split (Before/After)

| | ROC AUC | Precision@50 |
|---|---|---|
| BEFORE (naive random split) | 0.742 | 0.960 |
| AFTER (honest client-holdout split) | 0.748 | 0.820 |

ROC AUC stayed nearly identical between the two splits (0.742 vs 0.748),
which is reassuring — the model's general ranking ability doesn't appear to
depend on client leakage. However, Precision@50 dropped meaningfully under
the honest split (0.96 → 0.82), a 14-point gap. This is evidence that the
random split let the model partially memorize client-specific patterns
rather than learning a fully general rule — under random splitting, rows
from the same client can appear in both train and test, giving the model an
unfair shortcut. The client-holdout number (0.82) is the one I trust and
report going forward, since it reflects performance on genuinely unseen
clients — the real-world deployment scenario.

In [55]:
# Ensure target and baseline exist (safe to re-run, self-contained)
df['target'] = (df['trend_direction'].astype(str).str.lower() == 'down').astype(int)
df['baseline_score'] = (df.groupby('position_tier')['ctr'].transform('mean') - df['ctr']) * df['impressions_90d']

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# BEFORE: naive random split (not grouped by client)
train_idx_r, test_idx_r = train_test_split(df.index, test_size=0.2, random_state=42, stratify=df['target'])
X_train_r = df.loc[train_idx_r, feature_cols].fillna(0)
y_train_r = df.loc[train_idx_r, 'target']
X_test_r = df.loc[test_idx_r, feature_cols].fillna(0)
y_test_r = df.loc[test_idx_r, 'target']

rf_random = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25, class_weight='balanced_subsample', random_state=42)
rf_random.fit(X_train_r, y_train_r)
proba_random = rf_random.predict_proba(X_test_r)[:, 1]

print("=== BEFORE: naive random split ===")
print("ROC AUC:", round(roc_auc_score(y_test_r, proba_random), 3))
print("Precision@50:", round(precision_at_k(y_test_r, proba_random, 50), 3))

print()
print("=== AFTER: honest client-holdout split ===")
print("ROC AUC:", round(roc_auc_score(y_test, proba_honest), 3))
print("Precision@50:", round(precision_at_k(y_test, proba_honest, 50), 3))# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


=== BEFORE: naive random split ===
ROC AUC: 0.742
Precision@50: 0.96

=== AFTER: honest client-holdout split ===
ROC AUC: 0.748
Precision@50: 0.82


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage Audit

Re-running the leakage hunt from Week 3 on my final feature set (`ctr`,
`avg_position`, `impressions_90d`, `clicks_90d`, `engagement_rate`,
`scroll_rate`, `search_volume`, `content_age_days`):

- **Label-derived features:** `trend_direction` and `trend_pct` (the
  columns the label is built from) are confirmed absent from my real
  feature set (`'trend_direction' not in feature_cols and 'trend_pct' not
  in feature_cols` → True).
- **Product flags:** this dataset contains no product-decision columns at
  all (no health_score, needs_ctr_fix, is_quick_win, priority_score, or
  action_type), so none could have leaked in even by accident.
- **Proof the test harness works:** deliberately adding `trend_pct` back in
  as a feature pushed ROC AUC from 0.748 to a perfect 1.000 — the exact
  "too good to be true" signature the leakage checklist warns about. This
  confirms two things at once: my test harness correctly catches leakage
  when it's present, and my real model's honest score (0.748) is genuinely
  free of this kind of leakage, since removing the suspect column collapses
  the score straight back down to a normal, believable level.

**Conclusion:** the model's real, reportable performance is ROC AUC 0.748
under a client-holdout split — not an artifact of a leaky feature or a lax
split.

In [56]:
leaky_feature_cols = feature_cols + ['trend_pct']

X_train_leaky = df.loc[train_idx, leaky_feature_cols].fillna(0)
X_test_leaky = df.loc[test_idx, leaky_feature_cols].fillna(0)

rf_leaky = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25, class_weight='balanced_subsample', random_state=42)
rf_leaky.fit(X_train_leaky, y_train)
proba_leaky = rf_leaky.predict_proba(X_test_leaky)[:, 1]

print("WITHOUT trend_pct (honest, real model):", round(roc_auc_score(y_test, proba_honest), 3))
print("WITH trend_pct (deliberately leaky):", round(roc_auc_score(y_test, proba_leaky), 3))
print()
print("Confirmed absent from features:", 'trend_direction' not in feature_cols and 'trend_pct' not in feature_cols)

WITHOUT trend_pct (honest, real model): 0.748
WITH trend_pct (deliberately leaky): 1.0

Confirmed absent from features: True


In [57]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim Rewrite

**Original (overconfident) claim:**
"Random Forest predicts which pages are declining, beating the baseline by
3x."

**Rewritten (safe language):**
"In this dataset, under a client-holdout validation split, the Random
Forest model's top-50 ranked pages showed observed traffic decline in 82%
of cases, compared to 28% for the hand-written baseline rule. This is a
directional, decision-support signal for prioritizing review — not a causal
or guaranteed prediction of future performance, and it has not been tested
on clients or content types outside this dataset."

In [58]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [59]:
# No computation needed — this section is a language exercise.
# Confirms the exact numbers referenced in the rewritten claim above.
print("Random Forest precision_at_50 (client-holdout):", round(precision_at_k(y_test, proba_honest, 50), 3))
print("Baseline precision_at_50 (client-holdout):", round(precision_at_k(y_test, base_scores, 50), 3))

Random Forest precision_at_50 (client-holdout): 0.82
Baseline precision_at_50 (client-holdout): 0.28


In [60]:
print("Split strategy:", split_strategy)
print("Train/Test rows:", len(train_idx), "/", len(test_idx))
print("Honest ROC AUC:", round(roc_auc_score(y_test, proba_honest), 3))
print("Honest Precision@50:", round(precision_at_k(y_test, proba_honest, 50), 3))
print("Leaky-feature test ROC AUC:", round(roc_auc_score(y_test, proba_leaky), 3))
print("All checks match expected values:",
      split_strategy == "client_holdout" and
      round(roc_auc_score(y_test, proba_honest), 3) == 0.748)

Split strategy: client_holdout
Train/Test rows: 27675 / 2325
Honest ROC AUC: 0.748
Honest Precision@50: 0.82
Leaky-feature test ROC AUC: 1.0
All checks match expected values: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.